In [ ]:
import pandas as pd

from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import datetime as dt

import hssm
import hssm.plotting

%matplotlib inline
%config InlineBackend.figure_format='retina'

In [ ]:
feature_df = pd.read_csv("../feature_extraction/outdir/master_feature_matrix_compiled.csv")
feature_df.head()

In [ ]:
sample_df = feature_df.copy()
sample_df.head()

In [ ]:
sample_df = sample_df[["user_id", "generation", "vase", 
                       "is_selected", "is_parent_vase", "adjusted_selection_elapsed_time_s"]]
sample_df.head(10)

In [ ]:
def compress_to_ddm(df):
    def agg(g):
        selected_is_parent_vase = 1 if g["is_selected"].iloc[0] == 1 and g["is_parent_vase"].iloc[0] == True else -1
        rt = g["adjusted_selection_elapsed_time_s"].iloc[0]  # same for both rows
        response = -1 if g["is_selected"].iloc[0] == 1 and g["vase"].iloc[0] == "vase_1" else 1
        return pd.Series({"rt": rt, "response": response, "selected_is_parent_vase": selected_is_parent_vase})
    
    return (df.groupby(["user_id", "generation"])
              .apply(agg, include_groups=False)
              .reset_index())

ddm_df = compress_to_ddm(sample_df)
print(f"Compressed {len(sample_df)} rows to {len(ddm_df)} rows.")
ddm_df.head()


In [ ]:
# Check for any negative, zero values, or NaNs in the rt column
negative_or_zero_rt = ddm_df[ddm_df['rt'] <= 0]
print(f"Found {len(negative_or_zero_rt)} rows with negative or zero RT values.")
# Check for any NaN values in the rt column
nan_rt = ddm_df[ddm_df['rt'].isnull()]
print(f"Found {len(nan_rt)} rows with NaN RT values.")

In [ ]:
# Remove participants missing RT values
participants_with_nan_rt = ddm_df[ddm_df['rt'].isnull()]['user_id'].unique()
ddm_df = ddm_df[~ddm_df['user_id'].isin(participants_with_nan_rt)]
print(f"Removed {len(participants_with_nan_rt)} participants with NaN RT values. Remaining rows: {len(ddm_df)}.")

In [ ]:
ddm_df.to_csv("ddm_input.csv", index=False)

In [ ]:
hssm.set_floatX("float32")

In [ ]:
def fit_hierarchical_model(
    data,
    model_name,
    model="ddm",
    include=None,
    model_config=None,
    p_outlier=0.05,
    n_draws=1000,
    n_tune=1000,
    n_chains=4,
    target_accept=0.85,
    base_path=".",
):
    out_dir = Path(base_path) / f"{model_name}_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    out_dir.mkdir(parents=True, exist_ok=True)

    m = hssm.HSSM(data=data, model=model, include=include, model_config=model_config, p_outlier=p_outlier)

    sample_kwargs = dict(
        sampler="pymc",
        draws=n_draws,
        tune=n_tune,
        chains=n_chains,
        cores=n_chains,
    )
    # blackbox likelihoods (e.g. full_ddm) use Slice sampling — target_accept is NUTS-only
    if m.loglik_kind != "blackbox":
        sample_kwargs["target_accept"] = target_accept

    m.sample(**sample_kwargs)

    m.log_likelihood()

    traces = m.traces
    loo = az.loo(traces, pointwise=True)
    summary = az.summary(traces)
    n_divergences = int(traces.sample_stats.diverging.sum().item())

    traces.to_netcdf(out_dir / "traces.nc")
    summary.to_csv(out_dir / "summary.csv")

    metrics = pd.DataFrame({
        "loo_elpd": [loo.elpd_loo],
        "loo_se": [loo.se],
        "p_loo": [loo.p_loo],
        "n_divergences": [n_divergences],
        "model_path": [str(out_dir)],
    }, index=[model_name])
    metrics.to_csv(out_dir / "metrics.csv")

    print(f"\nSaved to: {out_dir}")
    print(f"LOO ELPD: {loo.elpd_loo:.2f} ± {loo.se:.2f} | Divergences: {n_divergences}")

    return m, metrics

In [ ]:
BASE_PATH = "ddm/"

# Accumulates traces and metrics across all fitted models for comparison
model_registry = {}  # model_name -> InferenceData (traces)
all_metrics = []     # list of metrics DataFrames, concatenated at the end

## Global Models Testing Architecture
- Basic DDM with a, z, t, v
- full_global DDM with a, z, t, v, sv, st, sz
- Intermediate DDM with a, z, t, v, sv

### Basic DDM

In [ ]:
basic_model, basic_metrics = fit_hierarchical_model(
    ddm_df,
    model_name="basic_ddm",
    base_path=BASE_PATH,
)
model_registry["basic_ddm"] = basic_model.traces
all_metrics.append(basic_metrics)

In [ ]:
basic_model.graph()

In [ ]:
basic_model.pymc_model

In [ ]:
basic_model.plot_trace()

## Full DDM

- Includes all base parameters (a, z, v, t) and all inter-trial parameters (sv, sz, st)

In [ ]:
full_global_model, full_global_metrics = fit_hierarchical_model(
    ddm_df,
    model_name="full_global_ddm",
    model="full_ddm",
    base_path=BASE_PATH,
    n_draws=500,
    n_tune=500
)
model_registry["full_global_ddm"] = full_global_model.traces
all_metrics.append(full_global_metrics)

In [ ]:
full_global_model.graph()

In [ ]:
full_global_model.plot_trace()

## Intermediate DDM
- Core DDM parameters with the addition of inter-trial drift rate variability

In [ ]:
inter_global_model, inter_global_metrics = fit_hierarchical_model(
    ddm_df,
    model_name="inter_global_ddm",
    model="full_ddm",
    model_config={
        "list_params": ["v", "sv", "a", "z", "t"],
        },
    base_path=BASE_PATH,
    n_draws=500,
    n_tune=500
)
model_registry["inter_global_ddm"] = inter_global_model.traces
all_metrics.append(inter_global_metrics)

## Model Comparison

In [ ]:
# Concatenated metrics table across all fitted models
metrics_table = pd.concat(all_metrics)
metrics_table

In [ ]:
# LOO-based model comparison (requires >= 2 models in registry)
az.compare(model_registry)

In [ ]:
metrics_table.to_csv(Path(BASE_PATH) / "model_comparison.csv")